In [ ]:
%load_ext autoreload
%autoreload 2

from helpers.benchmark_plotting import *

models = [
    'vggt',
    # 'vggt_ba',
    # 'vggt_ba_ref',
    # 'vggt_edge_canny_all_2000iter',
    # 'vggt_edge_canny_early_stop_loss',
    'vggt_edge_canny_final',
    'vggt_edge_canny_skip_mlp',
    'vggt_edge_canny_dino3'
    # 'vggt_edge_canny_final_no_stop',
    # 'vggt_edge_canny_free_vars_100iter',
    # 'vggt_edge_canny_mlp_100iter',
    # 'vggt_edge_canny_mlp_100iter_',

    # 'vggt_edge_canny_mlp_paramz_100_iter',
    # 'vggt_edge_canny_mlp_paramzk_100iter',
    # 'vggt_edge_canny_mlp_paramzk_toff_100iter',
    # 'vggt_edge_canny_w2_25',
    # 'vggt_edge_canny_w2_40'
]
thr = 5

# add a method to discard images with high error at the end, useful fo NVS and not relevant for AUC
# can I log cameras with their error vs gt?

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import json

# 1. Load Data from JSON
with open("benchmark_results.json", "r") as f:
    results = json.load(f)

# Extract Datasets
datasets = results['metadata']['datasets']

# Initialize containers for reconstruction
auc_data = {'Dataset': datasets}
time_data = {'Dataset': datasets}
custom_palette = {}
method_order = []

# Parse experiments to populate data structures
for exp in results['experiments']:
    method_name = exp['method']
    
    # Append data
    auc_data[method_name] = exp['auc_scores']
    time_data[method_name] = exp['time_scores']
    
    # Map colors and order
    custom_palette[method_name] = exp['color']
    method_order.append(method_name)

# 2. Reshape Data for Plotting (Pandas Melt)
df_auc = pd.DataFrame(auc_data).melt(
    id_vars='Dataset', 
    var_name='Method', 
    value_name='AUC@5'
)

df_time = pd.DataFrame(time_data).melt(
    id_vars='Dataset', 
    var_name='Method', 
    value_name='Time (ms)'
)

# 3. Setup Figure
# sns.set_theme(style="whitegrid") # Optional: Uncomment for grid lines
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(21, 6))

# --- Plot 1: AUC @ 5° ---
sns.barplot(
    data=df_auc, 
    x='Dataset', 
    y='AUC@5', 
    hue='Method', 
    width=0.95,
    hue_order=method_order, 
    ax=ax1, 
    palette=custom_palette, 
    saturation=1, 
    edgecolor=None, 
    linewidth=0
)

ax1.set_title('AUC @ 5° Accuracy (Higher is Better)', fontsize=12, fontweight='bold')
ax1.set_ylabel('Accuracy', fontsize=12, fontweight='bold')
ax1.set_ylim(50, 109) # Adjusted Y-limit slightly for better fit

# NEW: Annotate AUC bars
for p in ax1.patches:
    h = p.get_height()
    if h > 0:
        ax1.annotate(f'{h:.1f}', 
                     (p.get_x() + p.get_width() / 2., h),
                     ha='center', va='center', 
                     fontsize=10, fontweight='bold', 
                     xytext=(0, 8), textcoords='offset points')
        
# --- Plot 2: Inference Time ---
sns.barplot(
    data=df_time,            # Uses the reshaped time data
    x='Dataset', 
    y='Time (ms)', 
    hue='Method', 
    width=0.95,
    hue_order=method_order, 
    ax=ax2,                  # Plots on the right-hand chart (ax2)
    palette=custom_palette, 
    saturation=1, 
    edgecolor=None, 
    linewidth=0
)

ax2.set_title('Inference Time (s) (Lower is Better)', fontsize=12, fontweight='bold')
ax2.set_ylabel('Time (s)', fontsize=12, fontweight='bold')

# Annotate Time bars (putting the numbers on top of the bars)
for p in ax2.patches:
    h = p.get_height()
    if h > 0:
        ax2.annotate(f'{h:.0f}', 
                     (p.get_x() + p.get_width() / 2., h),
                     ha='center', va='center', 
                     fontsize=10, fontweight='bold', 
                     xytext=(0, 8), textcoords='offset points')

# 4. Final Formatting
plt.tight_layout()
# Place legend on the first plot, remove from second to avoid duplication
ax1.legend(title='Method', loc='upper left', frameon=True)
if ax2.get_legend():
    ax2.get_legend().remove()

plt.show()

## TerraSky3D Test set

In [ ]:
# w2 = 50
dataset = "terrasky3D"
df = read_results(dataset, "sparse_150", models, thr=thr)

In [ ]:
print(latexfy(df, caption="", label=f"tab:{dataset}"))

In [ ]:
plot_auc5_with_time(df, dataset, models, thr)

## Scannet++

In [ ]:
# w2 = 50
dataset = "scannetpp"
df = read_results(dataset, "sparse_150", models, thr=thr)

In [ ]:
print(latexfy(df, caption="", label=f"tab:{dataset}"))

In [ ]:
split = len(df) // 2 +1 
plot_auc5_with_time(df[:split], dataset, models, thr, ignore_time=True)
plot_auc5_with_time(df[split:], dataset, models, thr)

## MipNeRF360


In [ ]:
# MLP (x|dino3 cls) seems to improve a bit
dataset = "mipnerf360"
df = read_results(dataset, "sparse_150", models, thr=thr, 
        remove=["stump", "treehill"], round_to=1
    )
df.mean(numeric_only=True) # why?

In [ ]:
print(latexfy(df, caption="", label=f"tab:{dataset}"))

## MIPNeRF360 (NVS)

In [ ]:
path = "/home/mattia/Desktop/Repos/gaussian-splatting/ablation/"
df = read_results_nvs(nvs_results_path=path, exclude_metrics=()).round(2)
df

In [ ]:
# path = "/home/mattia/Desktop/Repos/gaussian-splatting/ablation/"
df = read_results_nvs(
    exclude_metrics=(),
    column_order=["GT", "VGGT", "VGGT+BA", "VGGT+BA (Ref)", "VGGT+EA"]
    ).round(3)
df

In [ ]:
format_dict = {col: ("{:.2f}" if "PSNR" in col[1] else "{:.3f}") for col in df.columns}
latex_str = df.style.format(format_dict).to_latex()
print(latex_str)

In [ ]:
.

## MRE

In [ ]:
import glob
paths = sorted(glob.glob("benchmarks/vggt_edge_canny/*/*/sparse/training_logs.json"))

mre_ea = {}
# read all logs and print mean and median reprojection error per each of them
for p in paths:
    with open(p, "r") as f:
        log = json.load(f)
    mean_error = log["mean_reproj_error"]
    median_error = log["median_reproj_error"]
    # print(f"Path: {p.split('/')[2]}/{p.split('/')[3]}")
    # print(f"Mean Reprojection Error: {mean_error:.4f}")
    # print(f"Median Reprojection Error: {median_error:.4f}\n")
    mre_ea[f"{p.split('/')[2]}/{p.split('/')[3]}"] = mean_error
mre_ea

In [ ]:
import pycolmap
import numpy as np

def compute_mre_pycolmap(reconstruction):
    total_error = 0.0
    total_observations = 0
    
    # Iterate through all images in the reconstruction
    for image_id, image in reconstruction.images.items():
        # Only process registered images
        if not image.has_pose:
            continue
            
        camera = reconstruction.cameras[image.camera_id]
        
        # Get 2D-3D correspondences
        for point2D in image.points2D:
            # Check if point2D has a valid 3D point
            if not point2D.has_point3D():
                continue
            
            # Check if point3D_id is valid (not empty string)
            if not point2D.point3D_id or point2D.point3D_id not in reconstruction.points3D:
                continue
                
            # Get the 3D point coordinates
            point3D = reconstruction.points3D[point2D.point3D_id]
            
            # Project 3D point to camera frame
            # Convert Rigid3d to 4x4 matrix and apply transformation
            world_point_homo = np.append(point3D.xyz, 1.0)  # Convert to homogeneous coordinates
            cam_point_homo = image.cam_from_world.matrix() @ world_point_homo
            cam_coords = cam_point_homo[:3]  # Extract 3D coordinates
            
            # Project using camera model
            reprojected_2d = camera.img_from_cam(cam_coords)
            
            # Calculate reprojection error (L2 norm in pixels)
            observed_2d = point2D.xy
            error = np.linalg.norm(observed_2d - reprojected_2d)
            
            total_error += error
            total_observations += 1
                
    # Compute Mean Reprojection Error
    if total_observations == 0:
        return 0.0, 0
        
    mre = total_error / total_observations
    return mre, total_observations

# Usage:
paths = sorted(glob.glob("benchmarks/vggt_ba/*/*/sparse"))
mre_ba = {}
for path in paths:
    rec = pycolmap.Reconstruction(path)
    scene = f"{path.split('/')[2]}/{path.split('/')[3]}"
    if rec is not None:
        mre_value, count = compute_mre_pycolmap(rec)
        mre_ba[scene] = mre_value

paths = sorted(glob.glob("benchmarks/vggt_ba_ref/*/*/sparse"))
mre_ba_ref = {}
for path in paths:
    rec = pycolmap.Reconstruction(path)
    scene = f"{path.split('/')[2]}/{path.split('/')[3]}"
    if rec is not None:
        mre_value, count = compute_mre_pycolmap(rec)
        mre_ba_ref[scene] = mre_value


In [ ]:
import pandas as pd
mre_df = pd.DataFrame({
    "BA": pd.Series(mre_ba),
    "BA Refined": pd.Series(mre_ba_ref),
    "EA": pd.Series(mre_ea)
})
mre_df.loc['mean'] = mre_df.mean()  
mre_df.to_csv("benchmarks/mre_comparison.csv")

In [ ]:
mre_df

## batch

In [ ]:
dataset = "scannetpp"
remove = ["stump", "treehill"]

base_target = f"/home/mattia/Desktop/datasets/{dataset}"
base_repo = "/home/mattia/Desktop/Repos/batchsfm/benchmarks"
runs = os.listdir(base_repo)

scenes = os.listdir(base_target)
df = {}

for run in [f"vggt_edge_canny_{i}" for i in range(1,10)]:
    try:
        df_ = eval_colmap_model_all_scenes(
                    target_path=base_target,
                    target_folder="sparse_150",
                    input_path=f"{base_repo}/{run}/{dataset}",
                    input_folder="sparse",
                    return_df=True,
                    thrs=[thr],
                    round_to=4,
                    verbose=False,
                )
        # exclude "stump" and "treehill", the re do mean
        for rem in remove:
            df_.drop(rem, inplace=True, errors="ignore")
        df_.loc["mean"] = df_.mean()
        df[run] = df_.loc["mean"]
    except:
        pass

df = pd.DataFrame(df).T.round(2)

In [ ]:
df.loc['mean'] = df.mean().round(2)
df

In [ ]:
df['auc@5'].std()

In [ ]:
import pycolmap
path = "/home/mattia/Desktop/datasets/mipnerf360"

for scene in os.listdir(path)[1:]:
    if scene.startswith("."):
        continue
    scene_path = os.path.join(path, scene)

    # and do not start with .
    if not os.path.isdir(scene_path):
        continue

    # for images in ["images", "images_2", "images_4", "images_8"]:
    #     os.makedirs(os.path.join(scene_path, images, "1"), exist_ok=True)
    #     for img in os.listdir(os.path.join(scene_path, images)):
    #         if img.endswith(".JPG"):
    #             # print(f"Moving {img} to {os.path.join(scene_path, images, '1', img)}")
    #            os.rename(
    #                os.path.join(scene_path, images, img),
    #                os.path.join(scene_path, images, "1", img)
    #            )

    rec = pycolmap.Reconstruction(os.path.join(scene_path, "sparse/0"))
    # add 1/ to all images
    for img in rec.images.values():
        img.name = os.path.join("1", img.name)

    rec.write_binary(os.path.join(scene_path, "sparse/0"))

In [ ]:
from pathlib import Path
path = "/home/mattia/Desktop/datasets/mipnerf360"

for scene in os.listdir(path)[1:]:
    if scene.startswith("."):
        continue
    scene_path = os.path.join(path, scene)

    # and do not start with .
    if not os.path.isdir(scene_path):
        continue

    path_gt = f"/home/mattia/Desktop/datasets/mipnerf360/{scene}/images"
    path_train = f"/home/mattia/Desktop/datasets/mipnerf360/{scene}/images_4_150"

    def get_relative_paths(root_dir):
        root = Path(root_dir)
        # The rglob('*') method finds all files and directories recursively
        # We use .is_file() to filter out the directories themselves
        return [str(p.relative_to(root)) for p in root.rglob('*') if p.is_file()]

    # Example usage:
    images_gt = get_relative_paths(path_gt)
    images_train = get_relative_paths(path_train)

    compute_diff = sorted(list(set(images_gt) - set(images_train)))
    print(scene, len(compute_diff))

    with open(f"/home/mattia/Desktop/datasets/mipnerf360/{scene}/sparse/0/test.txt", "w") as f:
        for item in compute_diff:
            f.write("%s\n" % item)

In [ ]:
rec.images[1].camera

In [ ]:
rec = pycolmap.Reconstruction("/home/mattia/Desktop/Repos/batchsfm/benchmarks/vggt_edge_canny/mipnerf360/bicycle/sparse")
rec.cameras

In [ ]:
rec.cameras

In [ ]:
import pycolmap
rec = pycolmap.Reconstruction("/home/mattia/Desktop/datasets/mipnerf360/bicycle/sparse/0")
rec.cameras

# timings breakdown

In [ ]:
import pandas as pd
import re
import glob

def parse_timing_logs(file_paths):
    """
    Parses timing log files and returns a consolidated DataFrame.
    """
    data_list = []

    # Regex pattern to capture the key and the numeric value
    # Matches: [Key Name]: [Float Value]
    pattern = re.compile(r"^(.*?):\s*([\d.]+)")

    for path in file_paths:
        file_data = {"file_path": path.split('/')[-3]}  # Store relative file path
        try:
            with open(path, 'r') as f:
                for line in f:
                    match = pattern.search(line)
                    if match:
                        key = match.group(1).strip()
                        value = float(match.group(2))
                        file_data[key] = value
            data_list.append(file_data)
        except Exception as e:
            print(f"Error reading {path}: {e}")

    # Create DataFrame
    df = pd.DataFrame(data_list)
    
    # Move 'file_path' to the first column for better readability
    cols = ['file_path'] + [c for c in df.columns if c != 'file_path']
    return df[cols]

import glob
df = parse_timing_logs(glob.glob("benchmarks/vggt_edge_canny/mipnerf360/*/sparse/timings.txt"))
df

In [ ]:
df = parse_timing_logs(glob.glob("benchmarks/vggt_ba/mipnerf360/*/sparse/timings.txt"))
df

In [ ]:
df = parse_timing_logs(glob.glob("benchmarks/vggt_ba_ref/mipnerf360/*/sparse/timings.txt"))
df

In [ ]:
bars = {
    "VGGT+BA":{
        "vggt": 34.86,
        "tracks_establishment": 56.75,
        "optimizing": 3.02,
        "total": 113.02,
    },
    "VGGT+Ba (Ref)":{
        "vggt": 34.86,
        "tracks_establishment": 213.17,
        "optimizing": 4.4,
        "total": 281.89,
    },
    "VGGT+EA":{
        "vggt": 34.86,
        "loading_data": 4.445935,
        "optimizing": 45.566308-5.6722,
        "total": 50+34.86,
    },
}

# for all key, do the sum except "total"
for key, value in bars.items():
    value["other"] = value["total"] - sum(v for k, v in value.items() if k != "total")

bars

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt


# Remove 'total' for plotting components
plot_data = {}
for category, segments in bars.items():
    plot_data[category] = {k: v for k, v in segments.items() if k != 'total'}

# Create DataFrame and fill missing values for specific segments with 0
df = pd.DataFrame(plot_data).T.fillna(0)

# Sort the categories by the sum of their segments
df['total_height'] = df.sum(axis=1)
df = df.sort_values(by='total_height')
df = df.drop(columns=['total_height'])

# Generate the stacked bar plot
ax = df.plot(kind='bar', stacked=True, figsize=(12, 7), rot=0)

plt.title('Cumulative Segment Breakdown per Category', fontsize=14)
plt.xlabel('Category', fontsize=12)
plt.ylabel('Time (s)', fontsize=12)
plt.legend(title='Segments', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(axis='y', linestyle='--', alpha=0.7)

plt.tight_layout()


In [ ]:

# Remove 'total' and prepare data
plot_data = {cat: {k: v for k, v in segs.items() if k != 'total'} for cat, segs in bars.items()}

# Create DataFrame
df = pd.DataFrame(plot_data).T.fillna(0)

# Normalize each row to sum to 100%
df_percent = df.div(df.sum(axis=1), axis=0) * 100

# Plotting
ax = df_percent.plot(kind='bar', stacked=True, figsize=(12, 7), rot=0, color=plt.cm.tab10.colors)

plt.title('Relative Segment Breakdown (Normalized to 100%)', fontsize=14)
plt.xlabel('Category', fontsize=12)
plt.ylabel('Percentage (%)', fontsize=12)
plt.legend(title='Segments', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.ylim(0, 100)

# Add percentage labels inside the bars
for p in ax.patches:
    height = p.get_height()
    if height > 3:  # Only label segments larger than 3%
        x, y = p.get_xy() 
        ax.text(x + p.get_width()/2, y + height/2, f'{height:.1f}%', 
                ha='center', va='center', fontsize=9, color='white', fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
import glob, json

paths = glob.glob("benchmarks/vggt_edge_canny_final/mipnerf360/*/sparse/training_logs.json")

steps = 0
for path in paths:
    with open(path, "r") as f:
        log = json.load(f)
    steps += log["steps_actual"]
    print(f"{path}: {log['steps_actual']} steps")

print("Average steps:", int(steps / len(paths)))


In [ ]:
log